# Chess Expert — Train on Colab

This notebook downloads grandmaster games, turns them into training samples, and trains the `ChessPolicyNet` on a GPU. When it's done you download **`chess_expert.pt`** and run the model (and make the demo GIF) **locally on CPU**.

**Before you run anything:**
1. `Runtime → Change runtime type →` set **Hardware accelerator = GPU** (a paid A100/L4 is ideal for a strong model) and, if available, **High-RAM**.
2. In the *Clone* cell below, set `REPO_URL` to your GitHub repo.

Everything here runs on Colab. Nothing in this notebook needs your laptop.

> **💡 Do a cheap dry run first.** Before spending money on a long GPU session, run this whole notebook once on a **free** GPU with `--max-games 200` in the *Parse* cell and `--epochs 2` in the *Train* cell. That proves the full chain — download → parse → train → checkpoint → legal self-play — works end-to-end for pennies. Then remove the caps and launch the real run.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install dependencies
Set `REPO_URL` to your repo (after you push it to GitHub).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/chess-expert.git"  # <-- edit me

import os
if not os.path.isdir("chess-expert"):
    !git clone $REPO_URL
%cd chess-expert
!pip -q install -r requirements.txt

## 3. Download grandmaster games
Pulls real GM archives (Carlsen, Kasparov, Fischer, Karpov, Anand, ...) and merges them into `data/gm_games.pgn`. Add more players by editing `scripts/download_data.sh`.

In [ ]:
!bash scripts/download_data.sh
!ls -lh data/gm_games.pgn

## 4. Parse PGN → training samples
Positions are stored as **uint8** and saved as a memory-mappable `positions.npy`, so training RAM stays flat even for millions of positions.

- The GM archives are already strong, so we keep **all** games (`--min-elo 0`).
- For a **strong** model, train on everything: leave `--max-games` off.
- For a quick first pass, add e.g. `--max-games 5000`.

In [ ]:
!python -m src.data --pgn data/gm_games.pgn --out data/samples --min-elo 0

!python -m src.train \
  --data data/samples \
  --out models/chess_expert.pt \
  --epochs 20 \
  --batch-size 4096 \
  --lr 1e-3 \
  --channels 128 --blocks 10
# Speed: TF32 + cuDNN autotune are ON automatically on GPU (stable, no risk).
#   - Mixed-precision (AMP) is OFF by default because it hangs on some GPUs.
#     Try adding  --amp on  for a possible extra speedup; if it stalls, drop it.
#   - If you hit out-of-memory, lower --batch-size (e.g. 2048).
#   - Watch val move-match: it usually plateaus by ~10-15 epochs, so you can stop early.

In [ ]:
!python -m src.train \
  --data data/samples \
  --out models/chess_expert.pt \
  --epochs 30 \
  --batch-size 1024 \
  --lr 1e-3 \
  --channels 128 --blocks 10 \
  --workers 2

## 6. Sanity check: play a full game, assert every move is legal

In [ ]:
!python -m src.play --checkpoint models/chess_expert.pt --plies 60

## 7. Download the trained model
Save `chess_expert.pt` into your local repo's `models/` folder, then run the demo locally (see the README).

In [ ]:
from google.colab import files
files.download('models/chess_expert.pt')

## Optional: make the demo GIF right here on Colab
You can also generate the self-play GIF on Colab and download it.

In [ ]:
!python -m demo.make_gif --checkpoint models/chess_expert.pt --out demo/self_play.gif --plies 60 --temperature 0.6
from google.colab import files
files.download('demo/self_play.gif')